# Treinamento da rede neural para prever atraso

Notebook reduzido ao carregamento e balanceamento dos dados, treinamento de uma MLP com `CrossEntropyLoss` e avaliação. Classes: `0` = sem atraso; `1` = atraso.

O balanceamento é feito separadamente em treino, validação e teste, em memória. A busca escolhe hiperparâmetros usando validação; o teste só é usado na avaliação final.

In [ ]:
from pathlib import Path
import itertools
import joblib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, precision_recall_curve,
    ConfusionMatrixDisplay, PrecisionRecallDisplay
)

SEMENTE = 7
torch.manual_seed(SEMENTE)

raiz = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/tratar_dados_ml.py").is_file())
dados = joblib.load(raiz / "Modelos" / "dados_ml.joblib")

def balancear(X, y, rng):
    y = np.asarray(y).ravel().astype(np.int64)
    grupos = [np.flatnonzero(y == classe) for classe in (0, 1)]
    n = min(len(grupos[0]), len(grupos[1]))
    indices = np.concatenate([rng.choice(g, n, replace=False) for g in grupos])
    rng.shuffle(indices)
    return np.asarray(X)[indices], y[indices]

rng = np.random.default_rng(SEMENTE)
particoes = {}
for nome in ("treino", "validacao", "teste"):
    y_original = np.asarray(dados["y"][nome]).ravel()
    print(f"{nome}: antes [0, 1]={np.bincount(y_original.astype(int), minlength=2).tolist()}")
    particoes[nome] = balancear(dados["X"][nome], y_original, rng)
    print(f"{nome}: balanceado [0, 1]={np.bincount(particoes[nome][1], minlength=2).tolist()} | X={particoes[nome][0].shape}")

X_treino, y_treino = particoes["treino"]
X_validacao, y_validacao = particoes["validacao"]
X_teste, y_teste = particoes["teste"]

## Busca de hiperparâmetros e treinamento

A grade abaixo tem 16 combinações de learning rate, batch size, épocas máximas e dropout. Cada tentativa usa parada antecipada pela Average Precision da validação; o notebook guarda o melhor checkpoint global. Como os dados de treino estão balanceados, a `CrossEntropyLoss` é usada sem pesos de classe adicionais.

In [ ]:
X_treino_t = torch.as_tensor(X_treino, dtype=torch.float32)
y_treino_t = torch.as_tensor(y_treino, dtype=torch.long)
X_validacao_t = torch.as_tensor(X_validacao, dtype=torch.float32)

LEARNING_RATES = [1e-4, 3e-4]
BATCH_SIZES = [128, 256]
EPOCAS_MAXIMAS = [30, 60]
DROPOUTS = [0.1, 0.2]
WEIGHT_DECAYS = [0.01]
PACIENCIA = 7
combinacoes = list(itertools.product(LEARNING_RATES, BATCH_SIZES, EPOCAS_MAXIMAS, DROPOUTS, WEIGHT_DECAYS))
print(f"Combinações: {len(combinacoes)}")

def criar_modelo(dropout):
    return nn.Sequential(
        nn.Linear(X_treino.shape[1], 128), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 2)
    )

melhor_ap_global = -1.0
melhor_config = None
melhor_estado = None
melhores_scores_validacao = None
historico_melhor = None
resultados_busca = []

for tentativa, (lr, batch_size, epocas_max, dropout, weight_decay) in enumerate(combinacoes, start=1):
    semente_tentativa = SEMENTE + tentativa
    torch.manual_seed(semente_tentativa)
    modelo_tentativa = criar_modelo(dropout)
    criterio = nn.CrossEntropyLoss()
    otimizador = torch.optim.AdamW(modelo_tentativa.parameters(), lr=lr, weight_decay=weight_decay)
    lotes = DataLoader(
        TensorDataset(X_treino_t, y_treino_t), batch_size=batch_size, shuffle=True,
        generator=torch.Generator().manual_seed(semente_tentativa)
)

    melhor_ap = -1.0
    sem_melhoria = 0
    historico_tentativa = []
    for epoca in range(1, epocas_max + 1):
        modelo_tentativa.train()
        perda_acumulada = 0.0
        for xb, yb in lotes:
            otimizador.zero_grad()
            perda = criterio(modelo_tentativa(xb), yb)
            perda.backward()
            otimizador.step()
            perda_acumulada += perda.item() * len(xb)

        modelo_tentativa.eval()
        with torch.no_grad():
            scores_val = torch.softmax(modelo_tentativa(X_validacao_t), dim=1)[:, 1].numpy()
        ap_val = average_precision_score(y_validacao, scores_val)
        historico_tentativa.append({
            "epoca": epoca, "perda_treino": perda_acumulada / len(y_treino_t),
            "ap_validacao": ap_val
        })

        if ap_val > melhor_ap:
            melhor_ap = ap_val
            melhor_epoca = epoca
            melhor_estado_tentativa = {k: v.detach().clone() for k, v in modelo_tentativa.state_dict().items()}
            melhores_scores_tentativa = scores_val.copy()
            sem_melhoria = 0
        else:
            sem_melhoria += 1
            if sem_melhoria >= PACIENCIA:
                break

    resultados_busca.append({
        "learning_rate": lr, "batch_size": batch_size, "epocas_maximas": epocas_max,
        "dropout": dropout, "weight_decay": weight_decay, "melhor_epoca": melhor_epoca,
        "average_precision_validacao": melhor_ap
    })
    print(f"[{tentativa}/{len(combinacoes)}] lr={lr:g}, batch={batch_size}, épocas={epocas_max}, dropout={dropout} | AP val={melhor_ap:.4f}")

    if melhor_ap > melhor_ap_global:
        melhor_ap_global = melhor_ap
        melhor_estado = {k: v.clone() for k, v in melhor_estado_tentativa.items()}
        melhores_scores_validacao = melhores_scores_tentativa.copy()
        historico_melhor = historico_tentativa.copy()
        melhor_config = {
            "learning_rate": lr, "batch_size": batch_size, "epocas_maximas": epocas_max,
            "dropout": dropout, "weight_decay": weight_decay, "melhor_epoca": melhor_epoca
        }

resultados_busca = pd.DataFrame(resultados_busca).sort_values("average_precision_validacao", ascending=False).reset_index(drop=True)
display(resultados_busca)
print("Melhor configuração:", melhor_config)

modelo = criar_modelo(melhor_config["dropout"])
modelo.load_state_dict(melhor_estado)
modelo.eval()

# Convergência da configuração escolhida.
fig, eixo_perda = plt.subplots(figsize=(8, 4.5), layout="constrained")
eixo_perda.plot([r["epoca"] for r in historico_melhor], [r["perda_treino"] for r in historico_melhor], color="#2563eb")
eixo_perda.set_xlabel("Época"); eixo_perda.set_ylabel("CrossEntropyLoss — treino", color="#2563eb")
eixo_ap = eixo_perda.twinx()
eixo_ap.plot([r["epoca"] for r in historico_melhor], [r["ap_validacao"] for r in historico_melhor], color="#15803d")
eixo_ap.set_ylabel("Average Precision — validação", color="#15803d")
eixo_perda.set_title("Convergência — melhor configuração"); eixo_perda.grid(alpha=0.2)
plt.show()

## Avaliação e gráficos

O limiar de classificação é escolhido na validação para maximizar F1, ignorando limiares que prevejam apenas uma classe. O mesmo limiar é aplicado ao teste. Se o teste colapsar em uma classe, a célula interrompe com diagnóstico em vez de imprimir uma matriz enganosa.

In [ ]:
def probabilidade_atraso(X):
    with torch.no_grad():
        logits = modelo(torch.as_tensor(X, dtype=torch.float32))
        return torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

prob_val = melhores_scores_validacao
precisoes, recalls, limiares = precision_recall_curve(y_validacao, prob_val)
f1_limiares = 2 * precisoes[:-1] * recalls[:-1] / (precisoes[:-1] + recalls[:-1] + 1e-12)
n_positivos = np.array([(prob_val >= t).sum() for t in limiares])
validos = (n_positivos > 0) & (n_positivos < len(y_validacao))
if not validos.any():
    raise RuntimeError("A rede não separa as classes na validação; verifique dados e treinamento.")
indices_validos = np.flatnonzero(validos)
idx_limiar = indices_validos[np.argmax(f1_limiares[indices_validos])]
limiar = float(limiares[idx_limiar])
print(f"Limiar escolhido na validação: {limiar:.4f}")

def avaliar(nome, y_real, scores):
    previsoes = (scores >= limiar).astype(np.int64)
    contagens = np.bincount(previsoes, minlength=2)
    if np.count_nonzero(contagens) < 2:
        raise RuntimeError(f"{nome}: só uma classe prevista {contagens.tolist()}; matriz de confusão não será apresentada.")
    print(f"\n{nome}: [previsto 0, 1]={contagens.tolist()} | AP={average_precision_score(y_real, scores):.4f}")
    print(f"Precisão={precision_score(y_real, previsoes, zero_division=0):.3f} | Recall={recall_score(y_real, previsoes, zero_division=0):.3f} | F1={f1_score(y_real, previsoes, zero_division=0):.3f}")
    print("Matriz de confusão [real nas linhas, previsão nas colunas]:")
    print(confusion_matrix(y_real, previsoes, labels=[0, 1]))
    return previsoes

previsoes_validacao = avaliar("Validação", y_validacao, prob_val)
prob_teste = probabilidade_atraso(X_teste)
previsoes_teste = avaliar("Teste", y_teste, prob_teste)

fig, eixos = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for eixo, nome, reais, prev in [(eixos[0], "Validação", y_validacao, previsoes_validacao), (eixos[1], "Teste", y_teste, previsoes_teste)]:
    ConfusionMatrixDisplay.from_predictions(reais, prev, labels=[0, 1], display_labels=["Sem atraso", "Atraso"], cmap="Blues", colorbar=False, values_format="d", ax=eixo)
    eixo.set_title(f"{nome} — limiar {limiar:.3f}")
plt.show()

fig, eixo = plt.subplots(figsize=(7, 5), layout="constrained")
PrecisionRecallDisplay.from_predictions(y_validacao, prob_val, name="Validação", ax=eixo)
PrecisionRecallDisplay.from_predictions(y_teste, prob_teste, name="Teste", ax=eixo)
eixo.axhline(y_validacao.mean(), color="gray", linestyle="--", label="Referência: proporção de atrasos")
eixo.set_title("Precision–Recall — rede neural"); eixo.legend(); plt.show()